# Nettoyagae et préparation des données pour le machine learning 

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

file_path = "../data/dataset-68f599a4c9b84581895311-6907667a3f0be705109390.csv"

df = spark.read.csv(file_path,header=True,inferSchema=True)
df.show(5)


+---------+----------+--------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+
|RowNumber|CustomerId| Surname|CreditScore|Geography|Gender|Age|Tenure|  Balance|NumOfProducts|HasCrCard|IsActiveMember|EstimatedSalary|Exited|
+---------+----------+--------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+
|        1|  15634602|Hargrave|        619|   France|Female| 42|     2|      0.0|            1|        1|             1|      101348.88|     1|
|        2|  15647311|    Hill|        608|    Spain|Female| 41|     1| 83807.86|            1|        0|             1|      112542.58|     0|
|        3|  15619304|    Onio|        502|   France|Female| 42|     8| 159660.8|            3|        1|             0|      113931.57|     1|
|        4|  15701354|    Boni|        699|   France|Female| 39|     1|      0.0|            2|        0|             0|       93826.63|

### Détection des doublons dans le jeu de données

In [3]:
from pyspark.sql.functions import col, count

dublicates = df.groupBy(df.columns).count()
dublicates.show()

+---------+----------+----------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+-----+
|RowNumber|CustomerId|   Surname|CreditScore|Geography|Gender|Age|Tenure|  Balance|NumOfProducts|HasCrCard|IsActiveMember|EstimatedSalary|Exited|count|
+---------+----------+----------+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+-----+
|       79|  15575185|   Bushell|        757|    Spain|  Male| 33|     5| 77253.22|            1|        0|             1|      194239.63|     0|    1|
|      396|  15807432|     Cheng|        645|  Germany|Female| 37|     2|136925.09|            2|        0|             1|      153400.24|     0|    1|
|      503|  15714485|   Udinese|        774|   France|  Male| 60|     5| 85891.55|            1|        1|             0|       74135.48|     1|    1|
|      746|  15640059|     Smith|        606|   France|  Male| 40|     5|      0.0|     

### Suppression des colonnes non pertinentes

In [4]:
colonnes_a_suprimer = ["RowNumber","CustomerId","Surname"]
df = df.drop(*colonnes_a_suprimer)
df.show()

+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+
|CreditScore|Geography|Gender|Age|Tenure|  Balance|NumOfProducts|HasCrCard|IsActiveMember|EstimatedSalary|Exited|
+-----------+---------+------+---+------+---------+-------------+---------+--------------+---------------+------+
|        619|   France|Female| 42|     2|      0.0|            1|        1|             1|      101348.88|     1|
|        608|    Spain|Female| 41|     1| 83807.86|            1|        0|             1|      112542.58|     0|
|        502|   France|Female| 42|     8| 159660.8|            3|        1|             0|      113931.57|     1|
|        699|   France|Female| 39|     1|      0.0|            2|        0|             0|       93826.63|     0|
|        850|    Spain|Female| 43|     2|125510.82|            1|        1|             1|        79084.1|     0|
|        645|    Spain|  Male| 44|     8|113755.78|            2|        1|             

In [5]:
df.filter(df.Balance==0).count()

3617

### Encodage des valeurs catégorielles avec StringIndexer et OneHotEncoder

In [6]:
from pyspark.ml.feature import StringIndexer,OneHotEncoder

indexer = StringIndexer(inputCol="gender", outputCol="gender_num")

df_indexed = indexer.fit(df).transform(df)

df_indexed = df_indexed.drop("gender").withColumnRenamed("gender_num", "gender")
df_indexed.show()

+-----------+---------+---+------+---------+-------------+---------+--------------+---------------+------+------+
|CreditScore|Geography|Age|Tenure|  Balance|NumOfProducts|HasCrCard|IsActiveMember|EstimatedSalary|Exited|gender|
+-----------+---------+---+------+---------+-------------+---------+--------------+---------------+------+------+
|        619|   France| 42|     2|      0.0|            1|        1|             1|      101348.88|     1|   1.0|
|        608|    Spain| 41|     1| 83807.86|            1|        0|             1|      112542.58|     0|   1.0|
|        502|   France| 42|     8| 159660.8|            3|        1|             0|      113931.57|     1|   1.0|
|        699|   France| 39|     1|      0.0|            2|        0|             0|       93826.63|     0|   1.0|
|        850|    Spain| 43|     2|125510.82|            1|        1|             1|        79084.1|     0|   1.0|
|        645|    Spain| 44|     8|113755.78|            2|        1|             0|     

In [7]:
indexer = StringIndexer(inputCol="Geography", outputCol="GeographyIndex")
df_indexed = indexer.fit(df).transform(df)
encoder = OneHotEncoder(inputCols=["GeographyIndex"], outputCols=["GeographyVec"],dropLast=False)
df_encoded = encoder.fit(df_indexed).transform(df_indexed)
df_final = df_encoded.drop("Geography", "GeographyIndex")
df_final.show()


+-----------+------+---+------+---------+-------------+---------+--------------+---------------+------+-------------+
|CreditScore|Gender|Age|Tenure|  Balance|NumOfProducts|HasCrCard|IsActiveMember|EstimatedSalary|Exited| GeographyVec|
+-----------+------+---+------+---------+-------------+---------+--------------+---------------+------+-------------+
|        619|Female| 42|     2|      0.0|            1|        1|             1|      101348.88|     1|(3,[0],[1.0])|
|        608|Female| 41|     1| 83807.86|            1|        0|             1|      112542.58|     0|(3,[2],[1.0])|
|        502|Female| 42|     8| 159660.8|            3|        1|             0|      113931.57|     1|(3,[0],[1.0])|
|        699|Female| 39|     1|      0.0|            2|        0|             0|       93826.63|     0|(3,[0],[1.0])|
|        850|Female| 43|     2|125510.82|            1|        1|             1|        79084.1|     0|(3,[2],[1.0])|
|        645|  Male| 44|     8|113755.78|            2| 

In [9]:
df_final.toPandas().to_csv("../data/db_final.csv", index=False)